# YOLO26m — Superbike Detector Training
**Notebook for training & evaluation. Once happy with results, weights go to the backend/frontend `.py` files.**

| Setting | Value | Why |
|---|---|---|
| Model | yolo26m | Best accuracy/speed balance |
| Batch | 8 | Fits RTX 3070 8GB VRAM |
| Workers | 4 | Keeps CPU cooler (~50-70%) |
| Cache | disk | Faster epochs after first, no RAM spike |
| Augmentation | all off | Roboflow already did 3× augmentation |
| **Thermal cooldown** | **auto** | **Pauses if CPU > threshold, then resumes** |

## 0 · Environment Check

In [1]:
import torch
from ultralytics import YOLO

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU      : {props.name}")
    print(f"VRAM     : {props.total_memory / 1e9:.1f} GB")

# Cap interop threads — workers=4 handles loading, this prevents extra CPU heat
torch.set_num_interop_threads(2)

PyTorch  : 2.9.1+cu130
CUDA     : True
GPU      : NVIDIA GeForce RTX 3070 Laptop GPU
VRAM     : 8.6 GB


## 1 · Thermal Cooldown Helper
Reads CPU temperature after every epoch. If it's above your threshold, training pauses until it cools down, then continues automatically.

**Install requirement (one time):**
```bash
pip install psutil
# Windows users also need:
pip install wmi
```

In [2]:
import time
import platform
import psutil

# ── Tweak these two numbers to your comfort ───────────────────────────────────
CPU_TEMP_LIMIT  = 85   # °C — pause training above this
CPU_TEMP_RESUME = 75   # °C — resume training once cooled to this
POLL_INTERVAL   = 5    # seconds between temperature checks while cooling
# ─────────────────────────────────────────────────────────────────────────────

def get_cpu_temp():
    """Return CPU package temperature in °C, or None if unreadable."""
    system = platform.system()

    # ── Windows ──────────────────────────────────────────────────────────────
    if system == "Windows":
        try:
            import wmi
            w = wmi.WMI(namespace="root\\OpenHardwareMonitor")
            sensors = w.Sensor()
            cpu_temps = [
                float(s.Value)
                for s in sensors
                if s.SensorType == "Temperature" and "CPU" in s.Name
            ]
            return max(cpu_temps) if cpu_temps else None
        except Exception:
            pass

        # Fallback: WMI MSAcpi_ThermalZoneTemperature (less accurate but no OHM needed)
        try:
            import wmi
            w = wmi.WMI(namespace="root\\wmi")
            zones = w.MSAcpi_ThermalZoneTemperature()
            if zones:
                # Value is in tenths of Kelvin
                return (zones[0].CurrentTemperature / 10.0) - 273.15
        except Exception:
            pass

    # ── Linux ─────────────────────────────────────────────────────────────────
    if system == "Linux":
        try:
            temps = psutil.sensors_temperatures()
            # Try common sensor names in order of reliability
            for key in ("coretemp", "k10temp", "zenpower", "cpu_thermal", "acpitz"):
                if key in temps:
                    readings = [t.current for t in temps[key]]
                    return max(readings)
        except Exception:
            pass

    # ── macOS ─────────────────────────────────────────────────────────────────
    if system == "Darwin":
        try:
            import subprocess, re
            out = subprocess.check_output(["sudo", "powermetrics", "-n", "1",
                                           "--samplers", "smc"], text=True, timeout=5)
            m = re.search(r"CPU die temperature:\s+([\d.]+)", out)
            if m:
                return float(m.group(1))
        except Exception:
            pass

    return None   # couldn't read — cooldown won't trigger


def thermal_cooldown_callback(trainer):
    """
    Ultralytics on_train_epoch_end callback.
    Called after every epoch. Blocks until CPU cools if over limit.
    """
    epoch = trainer.epoch + 1
    temp  = get_cpu_temp()

    if temp is None:
        # Can't read temp — print a reminder but don't block
        if epoch % 10 == 0:
            print(f"  [Thermal] Epoch {epoch}: CPU temp unreadable. "
                  "Install OpenHardwareMonitor (Windows) or check psutil sensors (Linux).")
        return

    print(f"  [Thermal] Epoch {epoch} done — CPU: {temp:.1f}°C", end="")

    if temp >= CPU_TEMP_LIMIT:
        print(f"  🌡️  Too hot! Pausing until ≤ {CPU_TEMP_RESUME}°C ...")
        while True:
            time.sleep(POLL_INTERVAL)
            temp = get_cpu_temp()
            if temp is None:
                break
            print(f"     Cooling... {temp:.1f}°C", end="\r")
            if temp <= CPU_TEMP_RESUME:
                print(f"\n  ✅  CPU cooled to {temp:.1f}°C — resuming training.")
                break
    else:
        print("  ✅")


# Quick temp sanity check
t = get_cpu_temp()
if t:
    print(f"Current CPU temp: {t:.1f}°C  (limit={CPU_TEMP_LIMIT}°C, resume={CPU_TEMP_RESUME}°C)")
else:
    print("⚠️  CPU temp unreadable on this system.")
    print("   Windows: install OpenHardwareMonitor and run it as Admin, then 'pip install wmi'")
    print("   Linux  : check 'psutil.sensors_temperatures()' returns values")

Current CPU temp: 82.4°C  (limit=85°C, resume=75°C)


## 2 · Load Model

In [3]:
model = YOLO("yolo26n.pt")
print(f"Loaded yolo26n — {sum(p.numel() for p in model.model.parameters())/1e6:.1f}M params")

# Register the thermal cooldown — fires after every epoch automatically
model.add_callback("on_train_epoch_end", thermal_cooldown_callback)
print("Thermal cooldown callback registered ✅")

Loaded yolo26n — 2.6M params
Thermal cooldown callback registered ✅


## 3 · Train

In [ ]:
results = model.train(
    data="data.yaml",
    imgsz=640,
    epochs=200,
    patience=75,          # early stop if no improvement for 75 epochs
    save=True,
    save_period=25,       # save checkpoint every 25 epochs
    project="../runs/train",
    name="superbike_detector",
    exist_ok=True,

    # ── Thermal-safe loading ──────────────────────────────────────────
    batch=8,              # fits RTX 3070 VRAM, GPU does the work not CPU
    workers=2,            # 4 data-loading threads — enough without CPU spike
    cache="disk",         # cache preprocessed images to disk

    # ── Training recipe ──────────────────────────────────────────────
    amp=True,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    label_smoothing=0.1,
    dropout=0.1,

    # ── Augmentation OFF — Roboflow already did 3× aug ────────────────
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    fliplr=0.0,
    flipud=0.0,
    degrees=0.0,
    shear=0.0,
    mosaic=0.0,
    mixup=0.0,
    translate=0.0,
)

In [4]:
model = YOLO("runs/runs/train/superbike_detector/weights/last.pt")

results = model.train(resume =True)

New https://pypi.org/project/ultralytics/8.4.83 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.80  Python-3.11.14 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs\runs\train\superbike_detector\weights\last.pt, momentum

KeyboardInterrupt: 

## 4 · Results Summary

In [ ]:
print("Training complete!")
print(f"Best mAP50   : {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"Best mAP50-95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")
print(f"Saved to     : ../runs/train/superbike_detector/")

## 5 · Validate Best Weights

In [ ]:
best_model = YOLO("../runs/train/superbike_detector/weights/best.pt")
val_results = best_model.val(data="data.yaml", imgsz=640, batch=8)

print(f"mAP50    : {val_results.box.map50:.4f}")
print(f"mAP50-95 : {val_results.box.map:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall   : {val_results.box.mr:.4f}")

## 6 · Quick Inference Test

In [ ]:
import matplotlib.pyplot as plt

TEST_IMAGE = "path/to/test_image.jpg"  # ← change this

best_model = YOLO("../runs/train/superbike_detector/weights/best.pt")
preds = best_model.predict(TEST_IMAGE, imgsz=640, conf=0.25)

result_img = preds[0].plot()
plt.figure(figsize=(12, 8))
plt.imshow(result_img[..., ::-1])
plt.axis("off")
plt.title(f"Detections: {len(preds[0].boxes)}")
plt.tight_layout()
plt.show()

for box in preds[0].boxes:
    cls  = int(box.cls)
    conf = float(box.conf)
    name = best_model.names[cls]
    print(f"  {name}: {conf:.2f}  bbox={box.xyxy[0].tolist()}")

## 7 · Resume Training (if interrupted)

In [ ]:
# Uncomment to resume
# resume_model = YOLO("../runs/train/superbike_detector/weights/last.pt")
# resume_model.add_callback("on_train_epoch_end", thermal_cooldown_callback)
# resume_model.train(resume=True)

---
## ✅ Next Steps
Once you're happy with the metrics above, copy `best.pt` to your backend:
```
cp ../runs/train/superbike_detector/weights/best.pt ../backend/weights/superbike_detector.pt
```
Then load it in your backend `.py` with:
```python
from ultralytics import YOLO
model = YOLO("weights/superbike_detector.pt")
```